# One-Hot Encoding

I compare a quick `pd.get_dummies` example with a full preprocessing pipeline on the Adult income dataset.


In [1]:
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)

## Load the data

I keep a few numeric and categorical columns so the encoded result stays easy to inspect. OpenML version 2 makes the data source reproducible.


In [2]:
# Pin the dataset version so future runs use the same definition.
adult = fetch_openml(name="adult", version=2, as_frame=True, parser="auto")
data = adult.frame[[
    "age",
    "workclass",
    "education",
    "sex",
    "hours-per-week",
    "occupation",
    "class",
]].dropna().reset_index(drop=True)

data = data.rename(columns={"class": "income"})
X = data.drop(columns="income")
y = data["income"].eq(">50K")

print(f"Rows: {len(data):,}")
display(data.head())
pd.DataFrame({
    "count": y.value_counts().sort_index(),
    "share": y.value_counts(normalize=True).sort_index().round(3),
})

Rows: 46,033


,age,workclass,education,sex,hours-per-week,occupation,income
0,25,Private,11th,Male,40,Machine-op-inspct,<=50K
1,38,Private,HS-grad,Male,50,Farming-fishing,<=50K
2,28,Local-gov,Assoc-acdm,Male,40,Protective-serv,>50K
3,44,Private,Some-college,Male,40,Machine-op-inspct,>50K
4,34,Private,10th,Male,30,Other-service,<=50K


,count,share
income,,
False,34611,0.752
True,11422,0.248


## Split before encoding

I split the raw data first. The encoder then learns categories from training data only and safely ignores unseen test categories.


In [3]:
# Split raw columns before fitting any preprocessing step.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE,
)

numeric_features = ["age", "hours-per-week"]
categorical_features = ["workclass", "education", "sex", "occupation"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
    ),
])

# Numeric and categorical columns need different transformations.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    verbose_feature_names_out=False,
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = pd.Series({
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
})
metrics.round(3)

accuracy             0.805
balanced_accuracy    0.678
roc_auc              0.827
dtype: float64

In [4]:
print(classification_report(y_test, y_pred, target_names=["<=50K", ">50K"]))

pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["actual_<=50K", "actual_>50K"],
    columns=["pred_<=50K", "pred_>50K"],
)

              precision    recall  f1-score   support

       <=50K       0.83      0.93      0.88      8653
        >50K       0.67      0.43      0.52      2856

    accuracy                           0.80     11509
   macro avg       0.75      0.68      0.70     11509
weighted avg       0.79      0.80      0.79     11509



,pred_<=50K,pred_>50K
actual_<=50K,8038,615
actual_>50K,1634,1222


In [5]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
# Inspect a few transformed rows to see the new indicator columns.
encoded_preview = pd.DataFrame(
    model.named_steps["preprocessor"].transform(X_train.head()),
    columns=feature_names,
    index=X_train.head().index,
)

encoded_preview.iloc[:, :12]

,age,hours-per-week,workclass_Federal-gov,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_State-gov,workclass_Without-pay,education_10th,education_11th,education_12th
36300,-0.041645,-0.079260,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
45317,0.715829,0.748289,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
24321,-1.480846,1.244818,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
33610,1.246061,-0.493034,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
11546,-0.799119,-0.493034,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


The model is intentionally simple because encoding is the main topic. Accuracy looks acceptable, but recall for the `>50K` class is much lower, so accuracy alone hides an important weakness.


## A small `get_dummies` example

`pd.get_dummies` is convenient for exploration. Integer labels should be converted to strings when they represent categories rather than quantities.


In [6]:
demo_df = pd.DataFrame(
    {
        "integer_feature": [0, 1, 2, 1],
        "categorical_feature": ["socks", "fox", "socks", "box"],
    }
)

demo_df["integer_feature"] = demo_df["integer_feature"].astype(str)
pd.get_dummies(demo_df, columns=["integer_feature", "categorical_feature"], dtype=int)

,integer_feature_0,integer_feature_1,integer_feature_2,categorical_feature_box,categorical_feature_fox,categorical_feature_socks
0,1,0,0,0,0,1
1,0,1,0,0,1,0
2,0,0,1,0,0,1
3,0,1,0,1,0,0


## What I learned

One-hot encoding belongs inside the model pipeline when I want repeatable training and prediction. I also need class-specific metrics when the target is imbalanced.
